# MLP를 train/valid에서 직접 학습하고 MLflow에 남기기

## 이번 질문

공식 승인은 Candidate B Random Forest입니다. 이 노트북은 sklearn `MLPClassifier`를 **학습 2,900행과 검증 600행만** 써서 직접 학습합니다. iteration마다 학습 손실과 검증 점수가 어떻게 바뀌는지 표와 그림으로 본 뒤, 교실 MLflow가 켜져 있으면 같은 숫자를 Run에 남깁니다.

패키지 로더를 쓰지 않습니다. YAML을 딕셔너리로 읽고, pandas로 CSV를 읽고, sklearn으로 학습하고, mlflow API로 기록합니다. 공식 승인, 봉인 평가, 3장 서빙은 바꾸지 않습니다.


## 먼저 예상

신경망을 여러 번 반복 학습하면 학습 손실은 보통 내려갑니다. 검증 점수도 같이 좋아질지, 아니면 학습만 좋아지고 검증은 멈출지 한 문장으로 적어 봅니다.

## 실행과 관측

### 1. 저장소와 YAML 설정 읽기

모델 크기와 반복 횟수는 과정 YAML에서 읽습니다. 수강생별 MLflow experiment 이름과 tracking URI 후보는 이 노트북의 상단 상수에서 관리합니다. `MLFLOW_EXPERIMENT_NAME`을 각 수강생에게 할당한 이름으로 바꾸면 Run을 분리할 수 있습니다.


In [ ]:
from pathlib import Path

import yaml

# 수강생별로 이 값을 바꿔 Run을 별도 experiment에 기록합니다.
MLFLOW_EXPERIMENT_NAME = "tta6-pve3"
MLFLOW_TRACKING_URIS = (
    "https://mlflow-tta6-pve3.apps.learn.mrml.dev",
    "http://localhost:5000",
)

# 1. 노트북을 labs/ 에서 열어도 저장소 루트를 찾는다.
ROOT = next(
    candidate
    for candidate in (Path.cwd(), *Path.cwd().parents)
    if (candidate / "pyproject.toml").is_file() and (candidate / "configs").is_dir()
)

# 2. 학생 MLP 설정과 공통 run 이름을 YAML에서 읽는다.
with (ROOT / "configs/model-v2/student-profiles.yaml").open(encoding="utf-8") as file:
    catalog = yaml.safe_load(file)
with (ROOT / "labs/run/development.yaml").open(encoding="utf-8") as file:
    tracking = yaml.safe_load(file)

profile = catalog["profiles"][0]
params = profile["params"]
candidate_tracking_uris = tuple(uri.rstrip("/") for uri in MLFLOW_TRACKING_URIS)
print("profile:", profile["name"])
print("model:", profile["kind"])
print("threshold:", profile["threshold"])
print("max_iter:", params["max_iter"])
print("experiment:", MLFLOW_EXPERIMENT_NAME)
print("run_name:", tracking["run_name"])
print("tracking URI candidates:", candidate_tracking_uris)


### 2. train과 valid만 열고 지문 확인하기

학습에 쓰는 파일이 과정이 선언한 파일과 같은지 SHA-256으로 확인합니다. 봉인 평가 역할과 운영 역할 파일은 열지 않습니다.


In [ ]:
import hashlib
import json

import pandas as pd

train_path = ROOT / tracking["paths"]["train"]
valid_path = ROOT / tracking["paths"]["valid"]
lineage_path = ROOT / tracking["paths"]["data_lineage"]

if not train_path.is_file() or not valid_path.is_file():
    raise FileNotFoundError(
        "학습/검증 파일이 없습니다. "
        "`uv run python labs/run/prepare_data.py`로 데이터를 준비합니다."
    )

lineage = json.loads(lineage_path.read_text(encoding="utf-8"))
train_declared = lineage["role_datasets"]["train"]
valid_declared = lineage["role_datasets"]["valid"]


def sha256(path: Path) -> str:
    return hashlib.sha256(path.read_bytes()).hexdigest()


train_hash = sha256(train_path)
valid_hash = sha256(valid_path)
assert lineage["revision"] == "v2"
assert train_hash == train_declared["sha256"]
assert valid_hash == valid_declared["sha256"]

train = pd.read_csv(train_path)
valid = pd.read_csv(valid_path)
assert len(train) == train_declared["rows"]
assert len(valid) == valid_declared["rows"]

# record_id, target 은 모델 입력이 아닙니다.
feature_cols = [column for column in train.columns if column not in {"record_id", "target"}]
X_train, y_train = train[feature_cols], train["target"].astype(int)
X_valid, y_valid = valid[feature_cols], valid["target"].astype(int)

pd.DataFrame(
    {
        "rows": [len(train), len(valid)],
        "features": [len(feature_cols), len(feature_cols)],
        "sha256_prefix": [train_hash[:12], valid_hash[:12]],
    },
    index=["train", "valid"],
)


### 3. sklearn MLP를 iteration마다 학습하기

전처리는 한 번만 맞춥니다. 그다음 `MLPClassifier`의 `max_iter`를 1, 2, 3... 으로 늘리며 같은 모델을 이어서 학습합니다. 매 반복마다 학습 손실과 검증 ROC-AUC를 표에 쌓습니다. sklearn의 `early_stopping`은 쓰지 않습니다. 과정 valid를 우리가 직접 채점하기 때문입니다.


In [ ]:
from warnings import catch_warnings, simplefilter

import matplotlib.pyplot as plt
from sklearn.exceptions import ConvergenceWarning
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# 1. 결측을 채우고 스케일만 맞춘다.
preprocess = Pipeline(
    [
        ("impute", SimpleImputer(strategy="median")),
        ("scale", StandardScaler()),
    ]
)
X_train_p = preprocess.fit_transform(X_train)
X_valid_p = preprocess.transform(X_valid)

# 2. YAML에서 읽은 설정으로 MLP를 만든다.
max_iter = int(params["max_iter"])
model = MLPClassifier(
    hidden_layer_sizes=tuple(params["hidden_layer_sizes"]),
    activation=params["activation"],
    solver=params["solver"],
    alpha=params["alpha"],
    learning_rate_init=params["learning_rate_init"],
    batch_size=params["batch_size"],
    early_stopping=False,
    warm_start=True,
    random_state=int(catalog["random_seed"]),
)

rows = []
with catch_warnings():
    simplefilter("ignore", ConvergenceWarning)
    for step in range(1, max_iter + 1):
        model.set_params(max_iter=step)
        model.fit(X_train_p, y_train)
        valid_score = model.predict_proba(X_valid_p)[:, 1]
        rows.append(
            {
                "step": step,
                "train_loss": float(model.loss_curve_[-1]),
                "valid_roc_auc": float(roc_auc_score(y_valid, valid_score)),
            }
        )

history = pd.DataFrame(rows)
display(history.tail())

fig, axes = plt.subplots(1, 2, figsize=(8, 3))
axes[0].plot(history["step"], history["train_loss"])
axes[0].set_title("train.loss")
axes[0].set_xlabel("iteration")
axes[1].plot(history["step"], history["valid_roc_auc"])
axes[1].set_title("valid.roc_auc")
axes[1].set_xlabel("iteration")
fig.tight_layout()
plt.show()


### 4. 사용할 수 있는 MLflow에 같은 숫자를 기록하기

상단 상수에 선언한 공개 MLflow, 로컬 MLflow 순서로 `/health`를 확인하고 첫 번째 정상 서버에 기록합니다. 어느 서버도 응답하지 않아도 위의 표와 그림은 그대로 해석합니다.


In [ ]:
import urllib.error
import urllib.request

import mlflow
from sklearn.pipeline import Pipeline

tracking_uri = None
for candidate_tracking_uri in candidate_tracking_uris:
    if not candidate_tracking_uri.startswith(("http://", "https://")):
        continue
    try:
        with urllib.request.urlopen(
            f"{candidate_tracking_uri}/health", timeout=3
        ) as response:
            if 200 <= int(response.status) < 300:
                tracking_uri = candidate_tracking_uri
                break
    except (urllib.error.URLError, TimeoutError, ValueError, OSError):
        continue

mlflow_ok = tracking_uri is not None

student_run_id = None
if not mlflow_ok:
    print("MLFLOW_NOT_RUNNING: 표와 그림만 보고, 공식 JSON으로 승인을 확인합니다.")
else:
    previous_uri = mlflow.get_tracking_uri()
    fitted = Pipeline([("preprocess", preprocess), ("model", model)])
    try:
        mlflow.set_tracking_uri(tracking_uri)
        mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)
        with mlflow.start_run(run_name=tracking["run_name"]) as run:
            mlflow.set_tags(
                {
                    "aiqa.profile": profile["name"],
                    "not_official_evidence": "true",
                }
            )
            mlflow.log_params(
                {
                    "model_kind": profile["kind"],
                    "threshold": profile["threshold"],
                    "max_iter": max_iter,
                    "train_data_hash": train_hash,
                    "valid_data_hash": valid_hash,
                }
            )
            for item in history.itertuples(index=False):
                mlflow.log_metric("train.loss", item.train_loss, step=int(item.step))
                mlflow.log_metric(
                    "valid.roc_auc", item.valid_roc_auc, step=int(item.step)
                )
            mlflow.sklearn.log_model(
                sk_model=fitted,
                name="model",
                serialization_format=mlflow.sklearn.SERIALIZATION_FORMAT_CLOUDPICKLE,
            )
            student_run_id = run.info.run_id
        print("LOGGED", student_run_id)
        print("UI: Model training /", MLFLOW_EXPERIMENT_NAME)
    finally:
        mlflow.set_tracking_uri(previous_uri)


## 해석과 기록

왼쪽 `train.loss`가 내려가는데 오른쪽 `valid.roc_auc`가 평평하거나 나빠지면 과적합입니다. 이 숫자는 개발 관찰입니다. 공식 승인은 여전히 Candidate B이고, 3장 서빙도 Candidate B를 유지합니다. 공식 Run과 학생 Run은 서로 다른 실행입니다.

## 결과 점검


In [ ]:
assert profile["name"] == "candidate-c"
assert profile["kind"] == "mlp_classifier"
assert len(history) == int(params["max_iter"])
assert set(history.columns) == {"step", "train_loss", "valid_roc_auc"}
assert list(history["step"]) == list(range(1, int(params["max_iter"]) + 1))
assert MLFLOW_EXPERIMENT_NAME.strip()
assert tracking["run_name"] == "student-mlp-train-valid"
print("학생 MLP iteration 표를 확인했습니다.")
if student_run_id:
    print("MLflow run:", student_run_id)
else:
    print("MLflow는 기록하지 않았습니다.")


## 다음 확인

MLflow 화면이 있으면 `Model training`에서 상단 `MLFLOW_EXPERIMENT_NAME`에 지정한 experiment를 엽니다. Metrics에서 `train.loss`와 `valid.roc_auc`가 iteration 축으로 보이는지 확인합니다. `GenAI`나 `Default` experiment는 이 Run이 아닙니다.

공식 판단은 `uv run python labs/run/model_status.py --revision v2`와 `docs/evidence/model-v2/release-manifest.json`으로 읽습니다. 이후 3장에서 떠 있는 모델이 Candidate B인지 따로 대조합니다.
